# V4.4.0 Seven Class: Choose and Evaluate Rejection Settings

This is an **experimental workflow**, not an adopted application model. Run the matching
V4.4.0 training notebook first. This notebook reads only that fresh, completed run; it
does not need earlier model weights, saved splits, masks, or threshold run folders.

The model returns probabilities for seven supported mango classes. Rejection settings
decide when those probabilities are too uncertain to show a disease result. This process
**does not train model weights** and does not turn unsupported photos into an eighth class.

The 200 unsupported photos are reused study material. The target assigns 100 to tuning
(choosing settings) and 100 to reporting (checking the frozen settings). They are not
new, untouched external evidence. Related captures must remain together. A fresh split
of reused photographs does not erase earlier use of those photographs.

Run the numbered steps in order. Review steps intentionally stop until completion of
their plain-English review tables. Keep completed runs: this notebook never overwrites
a frozen selection or starts a second evaluation in the same run folder.

**Outputs:** `My Drive/mangoleaf_experiments/v4.4.0/threshold_fitting/<unique-run-id>/`

**Input convention:** RGB float32 pixels from 0 to 255, resized to 224 × 224 with
bilinear interpolation. The trained model includes its own rescaling and its newly
fitted temperature is already folded into its weights. Do not apply either operation again.


## Step 1: Connect Drive and choose the input folders

**What this does:** Mounts Drive and prepares the same isolated scientific packages as the matching training workflow.

**Why this is needed:** The model must be read with compatible audited libraries, independently of Colab's provider packages.

**How this works:** Run both setup cells in Colab. The empty setup environment is created without ensurepip; pip --python installs only into that environment and requires provider pip 22.3 or newer. Host audit findings are preserved separately. All later code begins with %%mangoleaf and executes in the persistent isolated Python process. Edit the training folder name in the third cell. Conversion is not repeated because the completed training run already records it.

**Expected result:** The isolated interpreter and input paths are printed. Unreviewed worker advisories, skipped packages, or dependency conflicts stop fitting; host findings remain visible without modifying Colab libraries.


**Kernel startup:** The scientific process has a private IPython profile. Colab's configuration and automatic startup scripts are not loaded into that process. A startup failure names a local `kernel-startup.log`; connection credentials stay outside Drive evidence.

If environment preparation completed in the current session, it can be reused after a launcher repair: run the updated helper-definition cell, then run `start_notebook_kernel(RUNTIME_REVIEW)` in a new ordinary Colab code cell. Package preparation is not repeated. After startup succeeds, continue with the scientific-import and input-folder cell below.


In [ ]:
"""Prepare separate audited Python packages without changing Colab libraries.

Colab mounts Drive and forwards notebook output. Scientific cells execute in a
persistent isolated kernel; package isolation is not a separate security sandbox.
"""

import hashlib
import importlib.metadata
import json
import os
import site
import subprocess
import sys
import uuid
from datetime import datetime, timezone
from pathlib import Path


# New setup choices. Historical full Colab package snapshots are never installed.
SECURITY_PACKAGES = (
    "pip==26.2.1", "setuptools==84.0.0", "requests==2.34.2",
    "urllib3==2.8.0", "pip-audit==2.10.1",
)
TOOL_PACKAGES = (*SECURITY_PACKAGES, "uv==0.12.23")
TRAINING_PACKAGES = (
    *SECURITY_PACKAGES,
    "tensorflow[and-cuda]==2.19.0", "keras==3.15.1", "numpy==2.1.3",
    "Pillow==12.3.0", "ImageHash==4.3.2", "scipy==1.16.3",
    "matplotlib==3.11.2", "scikit-learn==1.9.1",
    "opencv-python-headless==5.0.0.93", "ipykernel==7.4.0",
    "ipython==9.17.1", "jupyter-client==8.10.0", "tornado==6.5.10",
)
CONVERTER_PACKAGES = (
    *(package for package in SECURITY_PACKAGES if not package.startswith("setuptools==")),
    # TensorFlow Hub 0.16.1 needs the real pkg_resources API removed in 82.
    # Its sdist-only advisory stays visible and gets a narrow operation review.
    "setuptools==81.0.0",
    "wheel==0.45.1",
    "tensorflow==2.19.0", "tf-keras==2.19.0",
    "tensorflow-decision-forests==1.12.0", "tensorflow-hub==0.16.1",
    # TFDF accepts newer YDF releases whose generated protos require Protobuf 6.
    # TensorFlow 2.19 requires Protobuf below 6; YDF 0.11 uses gencode 5.29.1.
    "ydf==0.11.0", "protobuf==5.29.6",
    "tensorflowjs==4.22.0", "keras==3.15.1", "Pillow==12.3.0",
    "numpy==2.1.3", "jax==0.4.35", "jaxlib==0.4.35",
    "flax==0.10.2", "packaging==23.2",
)


def runtime_write_json_once(path, value):
    """Preserve each attempt instead of overwriting earlier evidence."""
    with Path(path).open("x", encoding="utf-8") as handle:
        json.dump(value, handle, indent=2, allow_nan=False)
        handle.write("\n")


def runtime_package_versions(names):
    result = {}
    for name in names:
        try:
            result[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            result[name] = None
    return result


def runtime_clean_environment():
    """Prevent host Python paths and legacy Keras settings from leaking in."""
    environment = dict(os.environ)
    for key in ("PYTHONPATH", "PYTHONHOME", "VIRTUAL_ENV", "TF_USE_LEGACY_KERAS"):
        environment.pop(key, None)
    environment["PYTHONNOUSERSITE"] = "1"
    environment["TF_CPP_MIN_LOG_LEVEL"] = "2"
    return environment


def runtime_run_checked(command, log_path, environment=None):
    """Save full diagnostics. Never print credentials or a complete environment."""
    print("Runtime check:", Path(log_path).name, flush=True)
    result = subprocess.run(command, check=False, capture_output=True, text=True, env=environment)
    with Path(log_path).open("x", encoding="utf-8") as handle:
        handle.write(result.stdout or "")
        handle.write("\n")
        handle.write(result.stderr or "")
    if result.returncode != 0:
        diagnostic = Path(log_path)
        # A child preflight can fail in its own subprocess. Forward that named
        # log instead of making the outer traceback hide the actual failure.
        prefix = "RuntimeError: Runtime preparation stopped. Inspect the saved diagnostic log: "
        for line in reversed((result.stderr or "").splitlines()):
            if line.startswith(prefix):
                child = Path(line[len(prefix):])
                if child.parent.resolve() == diagnostic.parent.resolve() and child.is_file():
                    diagnostic = child
                break
        raise RuntimeError("Runtime preparation stopped. Inspect the saved diagnostic log: " + str(diagnostic))
    return result


def runtime_audit(python_path, output_directory, name, environment=None, paths=None, enforce=True, converter_build_review=False):
    """Worker findings block execution; provider findings remain explicit evidence."""
    report_path = Path(output_directory) / (name + "-audit.json")
    command = [str(python_path), "-m", "pip_audit", "--format", "json", "--output", str(report_path), "--progress-spinner", "off"]
    for path in paths or []:
        command.extend(["--path", str(path)])
    result = subprocess.run(command, check=False, capture_output=True, text=True, env=environment)
    with (Path(output_directory) / (name + "-audit.log")).open("x", encoding="utf-8") as handle:
        handle.write(result.stdout or "")
        handle.write("\n")
        handle.write(result.stderr or "")

    error = None
    dependencies = []
    try:
        report = json.loads(report_path.read_text(encoding="utf-8"))
        dependencies = report.get("dependencies", [])
        if not isinstance(dependencies, list) or not dependencies or not all(isinstance(row, dict) and row.get("name") for row in dependencies):
            raise ValueError("No complete installed-package inventory was returned.")
    except (OSError, ValueError, AttributeError) as failure:
        dependencies = []
        error = str(failure)

    findings = [{"name": row["name"], "version": row.get("version"), "vulnerabilities": row.get("vulns", [])} for row in dependencies if row.get("vulns")]
    skipped = [row for row in dependencies if row.get("skip_reason")]
    clean = not (error or findings or skipped or result.returncode != 0)
    reviewed = []
    blocking = []
    build_ids = {"GHSA-h35f-9h28-mq5c", "CVE-2026-59890", "PYSEC-2026-3447", "BIT-setuptools-2026-59890"}
    unpack_ids = {"GHSA-8rrh-rw8j-w5fx", "CVE-2026-24049"}
    for finding in findings:
        identities = [set([vulnerability.get("id"), *vulnerability.get("aliases", [])]) for vulnerability in finding["vulnerabilities"]]
        build_only = (
            converter_build_review and name == "converter"
            and finding["name"] == "setuptools" and finding["version"] == "81.0.0"
            and identities and all(identity and identity <= build_ids for identity in identities)
        )
        unpack_only = (
            converter_build_review and name == "converter"
            and finding["name"] == "wheel" and finding["version"] == "0.45.1"
            and identities and all(identity and identity <= unpack_ids for identity in identities)
        )
        if build_only:
            reviewed.append({
                "package": "setuptools", "version": "81.0.0",
                "advisory": "GHSA-h35f-9h28-mq5c",
                "status": "unresolved; affected sdist build/publish operation is absent",
                "reason": "The converter imports pkg_resources, installs wheels only, and converts SavedModels. It does not build or publish Python source distributions.",
                "source": "https://github.com/advisories/GHSA-h35f-9h28-mq5c",
            })
        elif unpack_only:
            reviewed.append({
                "package": "wheel", "version": "0.45.1",
                "advisory": "GHSA-8rrh-rw8j-w5fx",
                "status": "unresolved; affected wheel.cli.unpack API is absent",
                "reason": "The converter retains wheel 0.45.1. Package installation uses uv's installer, not wheel.cli.unpack. Conversion does not invoke wheel unpack or its vendored setuptools equivalent.",
                "source": "https://github.com/advisories/GHSA-8rrh-rw8j-w5fx",
            })
        else:
            blocking.append(finding)
    allowed = clean or (not error and not skipped and result.returncode == 1 and bool(reviewed) and not blocking)
    summary = {
        "status": "passed" if clean else "unresolved",
        "scope": "execution gate" if enforce else "provider host report; not a worker approval",
        "returnCode": result.returncode, "findings": findings, "skipped": skipped,
        "collectionError": error, "installedPackageCount": len(dependencies),
        "operationReviews": reviewed, "executionAllowed": allowed if enforce else None,
    }
    runtime_write_json_once(Path(output_directory) / (name + "-audit-summary.json"), summary)
    print("Dependency audit:", name, summary["status"], "affected packages:", len(findings), "skipped:", len(skipped), flush=True)
    if enforce and not allowed:
        raise RuntimeError("Dependency review is incomplete or contains advisories. Review the saved findings and install compatible fixes before repeating Step 1: " + str(report_path))
    if not enforce and not clean:
        print("Colab host findings remain unresolved. Report:", report_path, flush=True)
    if enforce and reviewed:
        print("Converter retains documented findings for unused sdist/unpack operations. Full reports and operation reviews:", report_path, flush=True)
    return summary


def runtime_check_environment(python_path, evidence, name, environment):
    runtime_run_checked([str(python_path), "-m", "pip", "check"], evidence / (name + "-pip-check.log"), environment)
    freeze = runtime_run_checked([str(python_path), "-m", "pip", "freeze", "--all"], evidence / (name + "-freeze.log"), environment)
    (evidence / (name + "-runtime-lock.txt")).write_text(freeze.stdout, encoding="utf-8")
    return runtime_audit(python_path, evidence, name, environment, converter_build_review=(name == "converter"))


def runtime_prepare_tools(environment_root, evidence, environment):
    """Install setup tools into an empty environment without provider ensurepip."""
    tools_root = Path(environment_root) / "tools"

    # Colab's ensurepip can fail before any experiment package is installed.
    # Create only the Python environment, without asking ensurepip to seed pip.
    runtime_run_checked(
        [sys.executable, "-m", "venv", "--without-pip", str(tools_root)],
        Path(evidence) / "tools-environment.log",
        environment,
    )
    tools_python = tools_root / ("Scripts/python.exe" if os.name == "nt" else "bin/python")

    # pip --python can install into an environment that has no pip yet.
    # The explicit interpreter targets tools_root, never Colab's site packages.
    runtime_run_checked(
        [sys.executable, "-m", "pip", "--python", str(tools_python),
         "install", "--only-binary", ":all:", "--upgrade", *TOOL_PACKAGES],
        Path(evidence) / "tools-package-setup.log",
        environment,
    )
    return tools_python


def prepare_reviewed_runtime(output_root, conversion=True):
    """Build real execution environments; never install into the Colab kernel."""
    output_root = Path(output_root).resolve()
    output_root.mkdir(parents=True, exist_ok=True)
    identifier = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
    evidence = output_root / ("runtime-" + identifier)
    evidence.mkdir(exist_ok=False)
    environment_root = Path("/content/mangoleaf-v440-runtime") / identifier
    environment_root.mkdir(parents=True, exist_ok=False)
    environment = runtime_clean_environment()

    # The small setup environment uses Colab's Python, but not its packages.
    # All subsequent installs target explicit environment Python executables.
    tools_python = runtime_prepare_tools(environment_root, evidence, environment)
    runtime_check_environment(tools_python, evidence, "tools", environment)

    # Audit the actual provider sites with a separate scanner. Keep skipped
    # provider wheels and host conflicts visible instead of upgrading Colab.
    host_paths = sorted({str(Path(path).resolve()) for path in [*site.getsitepackages(), site.getusersitepackages()] if Path(path).is_dir()})
    runtime_write_json_once(evidence / "host-inventory.json", {
        "python": sys.version,
        "paths": host_paths,
        "packages": sorted([{"name": dist.metadata.get("Name"), "version": dist.version} for dist in importlib.metadata.distributions()], key=lambda row: str(row["name"]).lower()),
    })
    host_check = subprocess.run([sys.executable, "-m", "pip", "check"], check=False, capture_output=True, text=True)
    (evidence / "host-pip-check.log").write_text((host_check.stdout or "") + "\n" + (host_check.stderr or ""), encoding="utf-8")
    host_summary = runtime_audit(tools_python, evidence, "host", environment, paths=host_paths, enforce=False)
    runtime_write_json_once(evidence / "host-review.json", {
        "audit": host_summary, "pipCheckReturnCode": host_check.returncode,
        "hostPackagesModified": False,
        "boundary": "Package isolation does not patch or sandbox the Colab host.",
    })

    # TFDF's released requirement is exactly TensorFlow 2.19.0. Python 3.11
    # supports that stack independently of the provider's kernel version.
    environment["UV_CACHE_DIR"] = str(environment_root / "uv-cache")
    environments = {"training": TRAINING_PACKAGES}
    if conversion:
        environments["converter"] = CONVERTER_PACKAGES
    interpreters = {}
    for name, packages in environments.items():
        folder = environment_root / name
        runtime_run_checked([str(tools_python), "-m", "uv", "venv", "--python", "3.11", "--managed-python", "--seed", str(folder)], evidence / (name + "-environment.log"), environment)
        python_path = folder / "bin" / "python"
        runtime_run_checked([str(tools_python), "-m", "uv", "pip", "install", "--only-binary", ":all:", "--python", str(python_path), *packages], evidence / (name + "-package-setup.log"), environment)
        runtime_check_environment(python_path, evidence, name, environment)
        interpreters[name] = python_path

    training_python = interpreters["training"]
    # Use the GPU libraries resolved with tensorflow[and-cuda], not mismatched
    # provider CUDA wheels. The machine's NVIDIA driver remains shared.
    nvidia_root = training_python.parent.parent / "lib/python3.11/site-packages/nvidia"
    cuda_paths = [str(path) for path in sorted(nvidia_root.glob("*/lib")) if path.is_dir()]
    cuda_bins = [str(path) for path in sorted(nvidia_root.glob("*/bin")) if path.is_dir()]
    training_environment = runtime_clean_environment()
    training_environment["LD_LIBRARY_PATH"] = os.pathsep.join([*cuda_paths, training_environment.get("LD_LIBRARY_PATH", "")])
    training_environment["PATH"] = os.pathsep.join([str(training_python.parent), *cuda_bins, training_environment.get("PATH", "")])
    training_environment["MPLBACKEND"] = "module://matplotlib_inline.backend_inline"

    # Both fitting and training prove the interpreter/package boundary first.
    isolation_probe = "import json,sys,site; assert sys.prefix != sys.base_prefix; assert not site.ENABLE_USER_SITE; print(json.dumps({'python':sys.executable,'prefix':sys.prefix,'paths':sys.path}))"
    runtime_run_checked([str(training_python), "-I", "-c", isolation_probe], evidence / "training-isolation-probe.log", training_environment)
    converter_python = interpreters.get("converter")
    if conversion:
        probe_file = evidence / "check-training-export.py"
        import inspect
        probe_source = "import hashlib, importlib.metadata, json, os, subprocess, sys\nfrom pathlib import Path\n"
        for helper in (runtime_write_json_once, runtime_package_versions, runtime_clean_environment, runtime_run_checked, runtime_conversion_probe):
            probe_source += "\n" + inspect.getsource(helper)
        probe_file.write_text(probe_source + "\nruntime_conversion_probe(Path(sys.argv[1]), Path(sys.argv[2]))\n", encoding="utf-8")
        runtime_run_checked([str(training_python), "-I", str(probe_file), str(evidence), str(converter_python)], evidence / "training-export-probe.log", training_environment)

    record = {
        "status": "isolated execution checks completed; inspect retained audit findings and operation reviews",
        "hostReview": "separate provider report; unresolved findings are not worker approval",
        "hostPackagesModified": False,
        "trainingPython": str(training_python), "trainingPackages": list(TRAINING_PACKAGES),
        "managedPythonRequest": "3.11",
        "converterPython": str(converter_python) if converter_python else None,
        "converterPackages": list(CONVERTER_PACKAGES) if conversion else [],
        "preflight": json.loads((evidence / "conversion-probe.json").read_text(encoding="utf-8")) if conversion else {"conversion": "not repeated for threshold fitting"},
        "artifacts": {path.relative_to(evidence).as_posix(): {"sha256": hashlib.sha256(path.read_bytes()).hexdigest(), "bytes": path.stat().st_size} for path in sorted(evidence.rglob("*")) if path.is_file()},
    }
    runtime_write_json_once(evidence / "runtime-preflight-complete.json", record)
    print("Isolated runtime prepared:", training_python, flush=True)
    return {
        "evidenceDirectory": evidence, "trainingPython": training_python,
        "trainingEnvironment": training_environment,
        "converterPython": converter_python,
        "converterCommand": converter_python.parent / "tensorflowjs_converter" if converter_python else None,
        "record": record,
    }


def runtime_conversion_probe(evidence, converter_python):
    """Check real GPU execution and cross-process export before any fitting."""
    evidence = Path(evidence)
    converter_python = Path(converter_python)
    converter_root = converter_python.parent.parent
    converter_environment = runtime_clean_environment()
    converter_environment.update(TF_USE_LEGACY_KERAS="1", CUDA_VISIBLE_DEVICES="-1")
    import numpy as np
    import tensorflow as tf

    # This uses the actual architecture but no pretrained weights and no fitting.
    # Conversion support must be checked before spending time on 15 training epochs.
    tf.keras.utils.set_random_seed(42)
    backbone = tf.keras.applications.MobileNetV3Large(
        input_shape=(224, 224, 3),
        include_top=False,
        weights=None,
        include_preprocessing=True,
    )
    backbone.trainable = False
    inputs = tf.keras.Input(shape=(224, 224, 3), dtype="float32", name="input_image")
    features = backbone(inputs, training=False)
    features = tf.keras.layers.GlobalAveragePooling2D(name="gap")(features)
    features = tf.keras.layers.Dropout(0.2, name="dropout")(features)
    logits = tf.keras.layers.Dense(7, name="logits")(features)
    outputs = tf.keras.layers.Activation("softmax", dtype="float32", name="probs")(
        logits
    )
    probe_model = tf.keras.Model(inputs, outputs, name="mangoleaf_v4_4_0_preflight")
    pixels = np.random.default_rng(42).uniform(
        0.0, 255.0, size=(1, 224, 224, 3)
    ).astype(np.float32)
    expected = probe_model(pixels, training=False).numpy()
    np.save(evidence / "preflight-input.npy", pixels, allow_pickle=False)
    np.save(evidence / "preflight-training-output.npy", expected, allow_pickle=False)
    saved_model = evidence / "preflight-saved-model"
    probe_model.export(str(saved_model), format="tf_saved_model", verbose=False)

    worker = evidence / "check-converter-saved-model.py"
    worker.write_text(
        "import sys\n"
        "from pathlib import Path\n"
        "import numpy as np\n"
        "import tensorflow as tf\n"
        "folder = Path(sys.argv[1])\n"
        "model = tf.saved_model.load(str(folder / 'preflight-saved-model'))\n"
        "pixels = np.load(folder / 'preflight-input.npy', allow_pickle=False)\n"
        "result = model.signatures['serving_default'](tf.constant(pixels))\n"
        "if len(result) != 1:\n"
        "    raise RuntimeError('Expected exactly one probability output.')\n"
        "probabilities = next(iter(result.values())).numpy()\n"
        "np.save(folder / 'preflight-converter-output.npy', probabilities, "
        "allow_pickle=False)\n",
        encoding="utf-8",
    )
    runtime_run_checked(
        [str(converter_python), str(worker), str(evidence)],
        evidence / "saved-model-compatibility.log",
        converter_environment,
    )
    actual = np.load(evidence / "preflight-converter-output.npy", allow_pickle=False)
    np.testing.assert_allclose(expected, actual, rtol=1e-4, atol=1e-5)

    graph_folder = evidence / "preflight-tfjs-model"
    runtime_run_checked(
        [
            str(converter_root / "bin" / "tensorflowjs_converter"),
            "--input_format=tf_saved_model",
            "--output_format=tfjs_graph_model",
            "--signature_name=serving_default",
            "--saved_model_tags=serve",
            str(saved_model),
            str(graph_folder),
        ],
        evidence / "preflight-conversion.log",
        converter_environment,
    )
    graph = json.loads((graph_folder / "model.json").read_text(encoding="utf-8"))
    if graph.get("format") != "graph-model" or not graph.get("weightsManifest"):
        raise RuntimeError("Converter did not produce a complete graph model.")

    for group in graph["weightsManifest"]:
        for relative in group["paths"]:
            shard = (graph_folder / relative).resolve()
            if not shard.is_relative_to(graph_folder.resolve()) or not shard.is_file():
                raise RuntimeError("A converter weight shard is missing or unsafe.")

        for weight in group["weights"]:
            if weight.get("quantization"):
                raise RuntimeError("The preflight graph unexpectedly uses quantization.")


    gpu_devices = tf.config.list_physical_devices("GPU")
    if not gpu_devices:
        raise RuntimeError("Select a Colab GPU runtime, then repeat Step 1. The isolated training process could not access a GPU.")
    with tf.device("/GPU:0"):
        gpu_check = tf.matmul(tf.ones((2, 2)), tf.ones((2, 2)))
    if "GPU" not in gpu_check.device.upper():
        raise RuntimeError("The training GPU check unexpectedly used a CPU.")

    runtime_write_json_once(evidence / "conversion-probe.json", {
        "trainingPython": sys.executable,
        "trainingPrefix": sys.prefix,
        "trainingVersions": runtime_package_versions(("tensorflow", "keras", "numpy", "Pillow", "scipy")),
        "gpuCount": len(gpu_devices),
        "gpuOperationDevice": gpu_check.device,
        "weights": "random initialization; weights=None",
        "trainingEpochs": 0,
        "savedModelMaximumAbsoluteDifference": float(np.max(np.abs(expected - actual))),
        "tfjsConversionCompleted": True,
        "browserProbabilityParity": "not measured by this preflight",
        "accuracy": "not measured by this preflight",
    })
    tf.keras.backend.clear_session()


"""Forward visible notebook cells to one persistent isolated Python kernel."""

import atexit
import json
import os
import sys
import uuid
from pathlib import Path
from queue import Empty

from IPython.display import clear_output, display
from jupyter_client import KernelManager
from jupyter_client.kernelspec import KernelSpecManager


def notebook_kernel_launcher():
    """Use the real IPython kernel with only this run's private configuration."""
    return '''from ipykernel.kernelapp import IPKernelApp


class MangoLeafKernelApp(IPKernelApp):
    def init_config_files(self):
        # IPython normally adds system/provider configuration directories.
        # This fresh profile belongs only to the prepared scientific process.
        self.config_file_paths = [self.profile_dir.location]

    def _run_startup_files(self):
        # The experiment has no automatic startup scripts. IPython normally
        # also searches system directories, even with a private IPYTHONDIR.
        return


if __name__ == "__main__":
    MangoLeafKernelApp.launch_instance()
'''


class NotebookKernel:
    """Keep variables between steps and return logs, figures, and failures."""

    def __init__(self, python_path, working_directory, environment):
        python_path = Path(python_path).absolute()
        bridge_root = Path(working_directory) / ("mangoleaf-bridge-" + uuid.uuid4().hex)
        specification_root = bridge_root / "kernels"
        specification = specification_root / "mangoleaf-v440"
        specification.mkdir(parents=True, exist_ok=False)
        profile_root = bridge_root / "ipython"
        profile_root.mkdir()
        launcher = bridge_root / "start-kernel.py"
        launcher.write_text(notebook_kernel_launcher(), encoding="utf-8")
        self.startup_log = bridge_root / "kernel-startup.log"
        kernel_environment = dict(environment)
        kernel_environment["IPYTHONDIR"] = str(profile_root)
        (specification / "kernel.json").write_text(json.dumps({
            "argv": [
                str(python_path), "-I", str(launcher), "-f", "{connection_file}",
                "--IPKernelApp.ipython_dir=" + str(profile_root),
                "--IPKernelApp.kernel_class=ipykernel.ipkernel.IPythonKernel",
            ],
            "display_name": "MangoLeaf isolated Python", "language": "python",
        }), encoding="utf-8")

        # Connection keys stay in local runtime storage, never in Drive evidence.
        self.manager = KernelManager(
            kernel_name="mangoleaf-v440",
            kernel_spec_manager=KernelSpecManager(kernel_dirs=[str(specification_root)]),
            connection_file=str(bridge_root / "connection.json"),
            ip="127.0.0.1",
            autorestart=False,
        )
        self.client = None
        self.ready = False
        self.running = False
        try:
            # Keep process startup output locally, apart from Drive evidence
            # and the connection file containing the authentication key.
            with self.startup_log.open("w", encoding="utf-8") as startup_output:
                self.manager.start_kernel(
                    cwd=str(working_directory), env=kernel_environment,
                    stdout=startup_output, stderr=startup_output,
                )
                self.client = self.manager.blocking_client()
                self.client.start_channels()
                self.client.wait_for_ready(timeout=120)
                self.ready = True
        except Exception as error:
            self.close()
            raise RuntimeError(
                "The isolated kernel could not start. Inspect its local startup log: "
                + str(self.startup_log)
            ) from error
        except BaseException:
            self.close()
            raise

    def close(self):
        """Stop the process so interrupted fitting cannot continue unseen."""
        self.ready = False
        try:
            if self.manager.has_kernel:
                self.manager.shutdown_kernel(now=True)
        finally:
            if self.client is not None:
                self.client.stop_channels()

    def execute(self, source):
        if not self.ready:
            raise RuntimeError("The isolated kernel is unavailable. Repeat Step 1, then restore the appropriate saved run.")
        if self.running:
            raise RuntimeError("An isolated cell is still running. Wait for its result before starting another step.")
        self.running = True
        failure = None
        try:
            request_id = self.client.execute(source, allow_stdin=False, stop_on_error=True)
            while True:
                try:
                    message = self.client.get_iopub_msg(timeout=1)
                except Empty:
                    if not self.manager.is_alive():
                        self.close()
                        raise RuntimeError("The isolated Python process stopped. Repeat Step 1 and use the documented recovery controls.")
                    continue
                # Ignore output from a previous request or kernel startup.
                if message.get("parent_header", {}).get("msg_id") != request_id:
                    continue
                kind = message["msg_type"]
                content = message["content"]
                if kind == "status" and content["execution_state"] == "idle":
                    break
                if kind == "stream":
                    target = sys.stderr if content["name"] == "stderr" else sys.stdout
                    target.write(content["text"])
                    target.flush()
                elif kind in ("display_data", "execute_result", "update_display_data"):
                    if kind == "update_display_data":
                        get_ipython().display_pub.publish(data=content["data"], metadata=content.get("metadata", {}), transient=content.get("transient", {}), update=True)
                    else:
                        display(content["data"], raw=True, metadata=content.get("metadata", {}), transient=content.get("transient", {}))
                elif kind == "clear_output":
                    clear_output(wait=content.get("wait", False))
                elif kind == "error":
                    failure = content
                    print("\n".join(content.get("traceback", [])), file=sys.stderr, flush=True)

            # The shell reply is checked as well, so an execution error cannot
            # turn into a successful green cell after its traceback is printed.
            while True:
                reply = self.client.get_shell_msg(timeout=30)
                if reply.get("parent_header", {}).get("msg_id") == request_id:
                    break
            if reply["content"].get("status") != "ok":
                failure = failure or reply["content"]
            if failure:
                raise RuntimeError("Isolated cell stopped: " + failure.get("ename", "execution error") + ": " + failure.get("evalue", "Inspect the traceback above."))
        except KeyboardInterrupt:
            self.close()
            raise RuntimeError("Execution interrupted; the isolated process was stopped. Repeat Step 1 and follow saved-run recovery before continuing.") from None
        except Empty:
            self.close()
            raise RuntimeError("The isolated kernel did not complete its reply. Repeat Step 1 before continuing.") from None
        finally:
            self.running = False


def start_notebook_kernel(review):
    """Register the explicit marker used by every scientific notebook cell."""
    previous = globals().get("_mangoleaf_runner")
    if previous is not None:
        previous.close()
    runner = NotebookKernel(review["trainingPython"], Path("/content"), review["trainingEnvironment"])
    globals()["_mangoleaf_runner"] = runner
    atexit.register(runner.close)

    def mangoleaf_cell(line, source):
        if line.strip():
            raise ValueError("The mangoleaf cell marker takes no arguments.")
        runner.execute(source)

    get_ipython().register_magic_function(mangoleaf_cell, magic_kind="cell", magic_name="mangoleaf")
    # Send only explicit paths and evidence, never a complete host environment.
    worker_record = {key: str(review[key]) if review[key] is not None else None for key in ("evidenceDirectory", "trainingPython", "converterPython", "converterCommand")}
    worker_record["record"] = review["record"]
    initialization = "import json, os, sys\nfrom pathlib import Path\nRUNTIME_REVIEW = json.loads(" + repr(json.dumps(worker_record)) + ")\n"
    initialization += "assert sys.prefix != sys.base_prefix, 'Expected isolated Python'\nassert str(Path(sys.executable).absolute()) == RUNTIME_REVIEW['trainingPython'], 'Wrong Python interpreter'\n"
    initialization += "RUNTIME_REVIEW['evidenceDirectory'] = Path(RUNTIME_REVIEW['evidenceDirectory'])\n"
    initialization += "RUNTIME_REVIEW['converterEnvironment'] = dict(os.environ)\nRUNTIME_REVIEW['converterEnvironment'].update(TF_USE_LEGACY_KERAS='1', CUDA_VISIBLE_DEVICES='-1')\n"
    initialization += "get_ipython().run_line_magic('matplotlib', 'inline')\nprint('Scientific cells use:', sys.executable)\n"
    try:
        runner.execute(initialization)
    except BaseException:
        runner.close()
        raise
    return runner


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

RUNTIME_REVIEW = prepare_reviewed_runtime(
    Path('/content/drive/MyDrive/mangoleaf_experiments/v4.4.0/runtime_preflight'),
    conversion=False,
)
start_notebook_kernel(RUNTIME_REVIEW)


In [ ]:
%%mangoleaf
import csv
import hashlib
import importlib.metadata
import importlib.util
import json
import math
import os
import random
import shutil
import subprocess
import sys
import uuid
import zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath


# EDIT THESE SETTINGS. Paste the completed fresh training folder name here.
EXPERIMENT_ROOT = Path('/content/drive/MyDrive/mangoleaf_experiments/v4.4.0')
TRAINING_RUN_NAME = 'PASTE_COMPLETED_V4_4_0_TRAINING_FOLDER_NAME'
OOD_ROOT = Path('/content/drive/MyDrive/mango_ood_test_v4_4_0')

SEED = 42
EXPECTED_OOD_IMAGES = 200
TARGET_TUNING_IMAGES = 100
TARGET_REPORTING_IMAGES = 100
BATCH_SIZE = 32

# Keep the agreed recipe. The confidence sweep is not part of this notebook.
CONFIDENCE_THRESHOLD = 0.70
MINIMUM_VALIDATION_RETENTION = 0.95
SIMILARITY_DISTANCE = 5
GRID = [number / 100 for number in range(101)]

EXPERIMENT_VERSION = 'v4.4.0'
CLASS_NAMES = [
    'anthracnose', 'bacterial_black_spot', 'bacterial_canker',
    'healthy_fruit', 'healthy_leaf', 'mango_scab', 'sooty_mold',
]
IMAGE_SIZE = (224, 224)
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.webp'}

training_argument = Path(TRAINING_RUN_NAME)
TRAINING_RUN = (
    training_argument if training_argument.is_absolute()
    else EXPERIMENT_ROOT / 'training' / training_argument
).resolve()
training_root = (EXPERIMENT_ROOT / 'training').resolve()
if TRAINING_RUN == training_root or training_root not in TRAINING_RUN.parents:
    raise ValueError('Choose a fresh completed run inside the V4.4.0 training folder.')

print('Training input:', TRAINING_RUN)
print('Unsupported-photo input:', OOD_ROOT)
print('Confidence remains:', CONFIDENCE_THRESHOLD)

# These imports run in the audited scientific process, never in Colab's kernel.
RUNTIME_AUDIT_DIR = RUNTIME_REVIEW['evidenceDirectory']
import imagehash
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from PIL import Image, ImageDraw, ImageOps
from IPython.display import display

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print('Fresh threshold runtime audit:', RUNTIME_AUDIT_DIR)


## Step 2: Define the file checks and record-writing helpers

**What this does:** Defines readable helpers for fingerprints, safe paths, and records that cannot be silently overwritten.

**Why this is needed:** A threshold must remain tied to the exact model, photos, and review decisions used to select it.

**How this works:** SHA-256 fingerprints detect changed bytes. Decoded fingerprints compare actual RGB content. Path checks prevent an archive from writing outside its scratch folder.

**Expected result:** The helper functions are available. No photos are changed.


In [ ]:
%%mangoleaf
def utc_now():
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def inside(root, relative):
    # Only ordinary relative POSIX paths are accepted from a manifest or ZIP.
    name = PurePosixPath(str(relative))
    if name.is_absolute() or '..' in name.parts or '\\' in str(relative) or ':' in str(relative):
        raise ValueError('Unsafe relative path: ' + str(relative))
    result = (Path(root) / Path(*name.parts)).resolve()
    if Path(root).resolve() not in result.parents:
        raise ValueError('A manifest path escaped its expected folder.')
    return result


def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def write_json_once(path, data):
    with Path(path).open('x', encoding='utf-8') as handle:
        json.dump(data, handle, indent=2, allow_nan=False)
        handle.write('\n')


def write_csv_once(path, rows, fields=None):
    rows = list(rows)
    fields = fields or list(rows[0])
    with Path(path).open('x', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)


def read_csv(path):
    with Path(path).open(newline='', encoding='utf-8-sig') as handle:
        return list(csv.DictReader(handle))


def fingerprint_image(path, relative):
    with Image.open(path) as picture:
        rgb = ImageOps.exif_transpose(picture).convert('RGB')
        rgb.load()
        decoded = hashlib.sha256(str(rgb.size).encode('utf-8') + rgb.tobytes()).hexdigest()
        perceptual = str(imagehash.phash(rgb, hash_size=8))
        dimensions = list(rgb.size)
    fingerprint = sha256_file(path)
    return {
        'id': fingerprint[:20], 'relativePath': relative,
        'sha256': fingerprint, 'bytes': Path(path).stat().st_size,
        'decodedRgbSha256': decoded, 'phash': perceptual,
        'dimensions': dimensions,
    }


def perceptual_distance(first, second):
    return (int(first, 16) ^ int(second, 16)).bit_count()


def package_versions():
    versions = {}
    for name in ['tensorflow', 'keras', 'numpy', 'Pillow', 'ImageHash', 'scipy', 'matplotlib']:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions


def verify_artifacts():
    if sha256_file(COMPLETE_PATH) != COMPLETE_HASH:
        raise RuntimeError('The training completion record changed.')
    for relative, expected in complete['artifacts'].items():
        artifact = inside(TRAINING_RUN, relative)
        if not artifact.is_file() or artifact.stat().st_size != expected['bytes']:
            raise RuntimeError('A completed training artifact is missing or changed: ' + relative)
        if sha256_file(artifact) != expected['sha256']:
            raise RuntimeError('A completed training artifact fingerprint changed: ' + relative)


def verify_photo(record, root):
    path = inside(root, record['relativePath'])
    if path.stat().st_size != record['bytes'] or sha256_file(path) != record['sha256']:
        raise RuntimeError('A reviewed photo changed: ' + record['relativePath'])
    return path


## Step 3: Verify the completed fresh model and unpack its saved split

**What this does:** Checks the completed training records and prepares read-only copies of its split for predictions.

**Why this is needed:** Earlier run folders and newly edited model files must not accidentally enter this experiment.

**How this works:** Verifies every training artifact, seven-class order, calibration metadata, and runtime versions. The ZIP is unpacked into a unique Colab scratch folder; source photos remain unchanged.

**Expected result:** A unique output run appears on Drive. Train, validation, and test counts are printed. No test predictions run.


In [ ]:
%%mangoleaf
COMPLETE_PATH = TRAINING_RUN / 'training_complete.json'
if not COMPLETE_PATH.is_file():
    raise FileNotFoundError('Finish the fresh V4.4.0 training notebook and paste its run folder in Step 1.')

complete = read_json(COMPLETE_PATH)
COMPLETE_HASH = sha256_file(COMPLETE_PATH)
if complete.get('experimentVersion') != EXPERIMENT_VERSION:
    raise ValueError('The input is not a completed V4.4.0 experiment.')
if complete.get('classNames') != CLASS_NAMES:
    raise ValueError('The training class order does not match the agreed seven classes.')
if complete.get('foldedTemperature') is not True:
    raise ValueError('The completed model must already include its fitted temperature.')
if complete.get('testPredictionsRun') is not False:
    raise ValueError('Mango test predictions must remain unused until rejection settings are frozen.')

required_artifacts = {
    'input_manifest.json', 'mango_inventory.json', 'scene_inventory.json',
    'dataset_split.zip', 'model_calibrated.keras', 'class_names.json',
    'temperature.json', 'runtime.json', 'plan.json',
    'validation_probabilities.npy', 'tfjs_model/model.json', 'tfjs_model/model_config.json',
}
if not required_artifacts.issubset(complete.get('artifacts', {})):
    raise ValueError('The completed training record lacks required fresh artifacts.')
verify_artifacts()

manifest = read_json(TRAINING_RUN / 'input_manifest.json')
if sha256_file(TRAINING_RUN / 'input_manifest.json') != complete['inputManifestSha256']:
    raise ValueError('The completed split fingerprint does not match its manifest.')
if read_json(TRAINING_RUN / 'class_names.json') != CLASS_NAMES:
    raise ValueError('The saved model class list differs.')

saved_runtime = read_json(TRAINING_RUN / 'runtime.json')
current_versions = package_versions()
for package in ['tensorflow', 'keras']:
    expected = saved_runtime['distributions'][package]
    if not expected or current_versions[package] != expected:
        raise RuntimeError('Use the same reviewed ' + package + ' version as the completed training run: ' + str(expected))

run_id = 'v4-4-0-thresholds-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8]
RUN_DIR = EXPERIMENT_ROOT / 'threshold_fitting' / run_id
RUN_DIR.mkdir(parents=True, exist_ok=False)
SCRATCH_DIR = Path('/content') / run_id
SPLIT_DIR = SCRATCH_DIR / 'dataset_split'
SPLIT_DIR.mkdir(parents=True, exist_ok=False)
FIGURE_DIR = RUN_DIR / 'figures'
FIGURE_DIR.mkdir()

mango_records = manifest['images']
if set(record['subset'] for record in mango_records) != {'train', 'val', 'test'}:
    raise ValueError('The fresh split must contain train, val, and test.')
expected_paths = set()
for record in mango_records:
    name = PurePosixPath(record['relativePath'])
    if record['label'] not in CLASS_NAMES or name.parts[:2] != (record['subset'], record['label']):
        raise ValueError('A mango record has an unexpected class or subset path.')
    inside(SPLIT_DIR, record['relativePath'])
    expected_paths.add(record['relativePath'])
if len(expected_paths) != len(mango_records):
    raise ValueError('The split contains repeated paths.')

with zipfile.ZipFile(TRAINING_RUN / 'dataset_split.zip') as archive:
    members = [member for member in archive.infolist() if not member.is_dir()]
    archive_names = [member.filename for member in members]
    if len(set(archive_names)) != len(archive_names) or set(archive_names) != expected_paths:
        raise ValueError('The ZIP image identities differ from the completed split.')
    expected_by_path = {record['relativePath']: record for record in mango_records}
    for member in members:
        destination = inside(SPLIT_DIR, member.filename)
        if member.file_size != expected_by_path[member.filename]['bytes']:
            raise ValueError('An archive member has an unexpected byte count.')
        destination.parent.mkdir(parents=True, exist_ok=True)
        with archive.open(member) as source, destination.open('xb') as target:
            shutil.copyfileobj(source, target)

for record in mango_records:
    verify_photo(record, SPLIT_DIR)

validation_records = [record for record in mango_records if record['subset'] == 'val']
test_records = [record for record in mango_records if record['subset'] == 'test']
if any(not any(record['label'] == label for record in validation_records) for label in CLASS_NAMES):
    raise ValueError('Every supported class needs validation examples.')

write_json_once(RUN_DIR / 'runtime.json', {
    'distributions': current_versions, 'pythonVersion': __import__('sys').version,
    'trainingRuntimeSha256': sha256_file(TRAINING_RUN / 'runtime.json'),
    'dependencyAuditDirectory': str(RUNTIME_AUDIT_DIR),
    'dependencyAuditCompleteSha256': sha256_file(RUNTIME_AUDIT_DIR / 'runtime-preflight-complete.json'),
    'dependencyAuditFiles': {
        path.relative_to(RUNTIME_AUDIT_DIR).as_posix(): {'sha256': sha256_file(path), 'bytes': path.stat().st_size}
        for path in sorted(RUNTIME_AUDIT_DIR.rglob('*')) if path.is_file()
    },
    'createdAt': utc_now(),
})
print('Fresh run output:', RUN_DIR)
print('Saved split counts:', dict(Counter(record['subset'] for record in mango_records)))
print('No mango test or OOD reporting predictions have run.')


## Step 4: Inventory unsupported photos and find possible overlap

**What this does:** Counts the merged OOD photos, checks that they open, and flags duplicate or related images.

**Why this is needed:** A photo used for training or background augmentation must not also pretend to be an independent rejection-check photo.

**How this works:** Checks bytes, decoded RGB content, and perceptual similarity against OOD, all raw mango records, and background records. Similarity is a review flag, not proof that two photos are duplicates.

**Expected result:** Inventory and editable review CSV files appear in the run folder. The count should be 200 before review.


In [ ]:
%%mangoleaf
if not OOD_ROOT.is_dir():
    raise FileNotFoundError('Correct OOD_ROOT to the actual merged 200-photo Drive folder.')

ood_records = []
unreadable = []
other_files = []
for path in sorted(OOD_ROOT.rglob('*')):
    if not path.is_file():
        continue
    relative = path.relative_to(OOD_ROOT).as_posix()
    if path.suffix.lower() not in IMAGE_EXTENSIONS:
        other_files.append(relative)
        continue
    try:
        record = fingerprint_image(path, relative)
        # A path-specific ID keeps exact copies independently reviewable.
        record['id'] = hashlib.sha256((relative + record['sha256']).encode()).hexdigest()[:20]
        ood_records.append(record)
    except Exception as error:
        unreadable.append({'relativePath': relative, 'error': str(error)})

if unreadable:
    raise ValueError('Unreadable photos need correction before continuing: ' + str(unreadable))
if len(ood_records) != EXPECTED_OOD_IMAGES:
    raise ValueError('Expected ' + str(EXPECTED_OOD_IMAGES) + ' OOD photos; found ' + str(len(ood_records)) + '. Correct the input folder or explicitly revise the agreed count.')

raw_mango_inventory = read_json(TRAINING_RUN / 'mango_inventory.json')['images']
scene_inventory = read_json(TRAINING_RUN / 'scene_inventory.json')['images']
external_records = [
    {'kind': kind, **record}
    for kind, records in [('mango', raw_mango_inventory), ('background', scene_inventory)]
    for record in records
]
pair_flags = []
for index, first in enumerate(ood_records):
    for second in ood_records[index + 1:]:
        exact = first['sha256'] == second['sha256']
        decoded_match = first['decodedRgbSha256'] == second['decodedRgbSha256']
        distance = perceptual_distance(first['phash'], second['phash'])
        if exact or decoded_match or distance <= SIMILARITY_DISTANCE:
            pair_flags.append({
                'firstId': first['id'], 'secondKind': 'ood', 'secondId': second['id'],
                'firstPath': first['relativePath'], 'secondPath': second['relativePath'],
                'exactBytes': str(exact).lower(), 'decodedMatch': str(decoded_match).lower(),
                'phashDistance': distance, 'relationship': 'REVIEW', 'notes': '',
            })
    for second in external_records:
        exact = first['sha256'] == second['sha256']
        decoded_match = first['decodedRgbSha256'] == second.get('decodedRgbSha256')
        distance = perceptual_distance(first['phash'], second['phash'])
        if exact or decoded_match or distance <= SIMILARITY_DISTANCE:
            external_relative = second.get('sourceRelativePath', second.get('relativePath', ''))
            external_identity = hashlib.sha256((second['kind'] + external_relative + second['sha256']).encode()).hexdigest()[:20]
            pair_flags.append({
                'firstId': first['id'], 'secondKind': second['kind'],
                'secondId': external_identity,
                'firstPath': first['relativePath'],
                'secondPath': external_relative,
                'exactBytes': str(exact).lower(), 'decodedMatch': str(decoded_match).lower(),
                'phashDistance': distance, 'relationship': 'REVIEW', 'notes': '',
            })

write_json_once(RUN_DIR / 'ood_inventory.json', {
    'sourceRoot': str(OOD_ROOT), 'images': ood_records,
    'nonImageFiles': other_files, 'readableCount': len(ood_records),
    'reuseStatus': 'Reused unsupported study photos; not a pristine external holdout.',
})
write_csv_once(RUN_DIR / 'ood_review.csv', [
    {'id': record['id'], 'relativePath': record['relativePath'],
     'keep': 'REVIEW', 'unsupported': 'REVIEW', 'captureGroup': '', 'sourceNotes': '', 'notes': ''}
    for record in ood_records
])
write_csv_once(RUN_DIR / 'similarity_review.csv', pair_flags, fields=[
    'firstId', 'secondKind', 'secondId', 'firstPath', 'secondPath',
    'exactBytes', 'decodedMatch', 'phashDistance', 'relationship', 'notes',
])
print('Readable OOD photos:', len(ood_records))
print('Similarity pairs needing review:', len(pair_flags))
print('Edit:', RUN_DIR / 'ood_review.csv')
print('Edit:', RUN_DIR / 'similarity_review.csv')


## Step 5: Review the photo content and related captures

**What this does:** Shows contact sheets and explains exactly how to complete the two review tables.

**Why this is needed:** File fingerprints cannot decide whether a photo belongs outside all seven supported classes or whether two captures show the same subject.

**How this works:** Inspect every photo. In ood_review.csv use keep=yes/no, unsupported=yes/no, and meaningful sourceNotes. Use the same captureGroup for known related captures. In similarity_review.csv use same_capture or unrelated and explain the decision.

**Expected result:** Finish both CSV reviews before Step 6. Do not decide based on model predictions.


**Review rules:**

- `unsupported=yes` means the photo does not belong to any of the seven supported mango classes.
  A supported mango image does not become OOD because it is hard to predict. Mark ambiguous
  photographs `keep=no` until their intended scope is resolved.
- Different views, crops, compressed copies, and bursts of the same subject belong in one
  `captureGroup`. For unrelated photos, leave that field empty.
- Exact or decoded copies may not survive as two retained photos. Keep one and exclude the other.
- OOD overlap with mango training material or background material must be excluded. A visual
  similarity flag may be unrelated; document that decision rather than deleting it automatically.
- If exclusions reduce the valid pool below 200, stop and revise the agreed counts or collect
  replacements. This notebook will not manufacture extra images to reach 100/100.
- Source notes may be plain English. Record unknown origins honestly.

The contact sheets are previews. Open full-size files when a small preview cannot establish scope.
Rerun Step 6 after editing the CSV files. Step 4 creates a fresh inventory and must not be rerun
inside the same run folder.


In [ ]:
%%mangoleaf
def show_contact_sheets(records, root, destination, per_page=25):
    destination.mkdir(parents=True, exist_ok=True)
    for start in range(0, len(records), per_page):
        page = records[start:start + per_page]
        canvas = Image.new('RGB', (1250, 1100), 'white')
        draw = ImageDraw.Draw(canvas)
        for offset, record in enumerate(page):
            left = (offset % 5) * 250
            top = (offset // 5) * 220
            with Image.open(inside(root, record['relativePath'])) as picture:
                thumbnail = ImageOps.exif_transpose(picture).convert('RGB')
                thumbnail.thumbnail((238, 174))
                canvas.paste(thumbnail, (left + 6, top + 4))
            draw.text((left + 6, top + 180), record['id'], fill='black')
            draw.text((left + 6, top + 197), Path(record['relativePath']).name[:34], fill='black')
        output = destination / ('ood_' + str(start // per_page + 1).zfill(2) + '.jpg')
        canvas.save(output, quality=92)
        display(canvas)


show_contact_sheets(ood_records, OOD_ROOT, RUN_DIR / 'review_contact_sheets')
print('Complete the review tables before continuing. No model predictions were used for this review.')


## Step 6: Freeze separate tuning and reporting groups

**What this does:** Validates the reviews and assigns whole related-photo groups to the two OOD subsets.

**Why this is needed:** A photo or related capture must not help select a threshold and also appear in its reporting group.

**How this works:** Uses a saved random seed and exact group-size balancing. If 100/100 cannot be achieved without splitting a related group, the cell stops with the group sizes so the targets can be deliberately revised.

**Expected result:** evaluation_inputs.json and an assignment CSV are frozen with fingerprints. They include 100 tuning and 100 reporting photos when the reviewed groups permit it.


In [ ]:
%%mangoleaf
def assign_whole_groups(groups, target, seed):
    # A subset-sum search finds an exact count while keeping every group intact.
    ordered = sorted(groups)
    random.Random(seed).shuffle(ordered)
    reachable = {0: []}
    for group in ordered:
        size = len(groups[group])
        additions = {}
        for count, chosen in list(reachable.items()):
            new_count = count + size
            if new_count <= target and new_count not in reachable and new_count not in additions:
                additions[new_count] = chosen + [group]
        reachable.update(additions)
    if target not in reachable:
        raise ValueError('Cannot assign exactly ' + str(target) + ' tuning images without splitting groups. Group sizes: ' + str(sorted(len(rows) for rows in groups.values())))
    return set(reachable[target])


photo_review = read_csv(RUN_DIR / 'ood_review.csv')
similarity_review = read_csv(RUN_DIR / 'similarity_review.csv')
reviews_by_id = {row['id']: row for row in photo_review}
if len(reviews_by_id) != len(photo_review) or set(reviews_by_id) != {row['id'] for row in ood_records}:
    raise ValueError('Review IDs must match the complete OOD inventory exactly.')
for record in ood_records:
    review = reviews_by_id[record['id']]
    if review['relativePath'] != record['relativePath']:
        raise ValueError('A reviewed filename was changed.')
    if review['keep'] not in {'yes', 'no'} or review['unsupported'] not in {'yes', 'no'}:
        raise ValueError('Finish yes/no decisions for: ' + record['relativePath'])
    if review['keep'] == 'yes' and (review['unsupported'] != 'yes' or not review['sourceNotes'].strip()):
        raise ValueError('Kept photos need confirmed unsupported scope and honest source notes: ' + record['relativePath'])

retained = [record for record in ood_records if reviews_by_id[record['id']]['keep'] == 'yes']
retained_ids = {record['id'] for record in retained}
parent = {record['id']: record['id'] for record in retained}


def find_group(identity):
    while parent[identity] != identity:
        parent[identity] = parent[parent[identity]]
        identity = parent[identity]
    return identity


def join_group(first, second):
    parent[find_group(second)] = find_group(first)


manual_groups = defaultdict(list)
for record in retained:
    label = reviews_by_id[record['id']]['captureGroup'].strip()
    if label:
        manual_groups[label].append(record['id'])
for identities in manual_groups.values():
    for identity in identities[1:]:
        join_group(identities[0], identity)

expected_pairs = {(row['firstId'], row['secondKind'], row['secondId']): row for row in pair_flags}
received_pairs = {(row['firstId'], row['secondKind'], row['secondId']): row for row in similarity_review}
if len(received_pairs) != len(similarity_review) or set(received_pairs) != set(expected_pairs):
    raise ValueError('The similarity review must include every original flagged pair exactly once.')
for key, pair in received_pairs.items():
    original = expected_pairs[key]
    if any(str(pair[field]) != str(original[field]) for field in ['firstPath', 'secondPath', 'exactBytes', 'decodedMatch', 'phashDistance']):
        raise ValueError('A flagged pair identity was edited. Only relationship and notes may change.')
    if pair['relationship'] not in {'same_capture', 'unrelated'} or not pair['notes'].strip():
        raise ValueError('Finish the similarity decision and explanation for: ' + pair['firstPath'])
    first_kept = pair['firstId'] in retained_ids
    second_kept = pair['secondKind'] == 'ood' and pair['secondId'] in retained_ids
    identical = pair['exactBytes'] == 'true' or pair['decodedMatch'] == 'true'
    if first_kept and (second_kept or pair['secondKind'] != 'ood') and identical:
        raise ValueError('Exclude duplicate/overlapping content before splitting: ' + pair['firstPath'])
    if first_kept and pair['secondKind'] != 'ood' and pair['relationship'] == 'same_capture':
        raise ValueError('An OOD capture overlaps mango/background material. Exclude: ' + pair['firstPath'])
    if first_kept and second_kept and pair['relationship'] == 'same_capture':
        join_group(pair['firstId'], pair['secondId'])

if len(retained) != TARGET_TUNING_IMAGES + TARGET_REPORTING_IMAGES:
    raise ValueError('Reviewed OOD pool contains ' + str(len(retained)) + ' photos, but the explicit targets total ' + str(TARGET_TUNING_IMAGES + TARGET_REPORTING_IMAGES) + '. Resolve exclusions or revise the targets before freezing.')

groups = defaultdict(list)
for record in retained:
    groups[find_group(record['id'])].append(record)
tuning_groups = assign_whole_groups(groups, TARGET_TUNING_IMAGES, SEED)
assigned_records = []
for identity, group in sorted(groups.items()):
    subset = 'ood_tune' if identity in tuning_groups else 'ood_report'
    for record in group:
        assigned_records.append({**record, 'groupId': identity, 'subset': subset})

assignment_counts = Counter(record['subset'] for record in assigned_records)
if assignment_counts != Counter({'ood_tune': TARGET_TUNING_IMAGES, 'ood_report': TARGET_REPORTING_IMAGES}):
    raise ValueError('The final assignments do not match the explicit targets.')
for record in ood_records:
    verify_photo(record, OOD_ROOT)
verify_artifacts()

evaluation_inputs = {
    'experimentVersion': EXPERIMENT_VERSION, 'trainingRun': str(TRAINING_RUN),
    'trainingCompleteSha256': COMPLETE_HASH, 'seed': SEED,
    'oodRoot': str(OOD_ROOT), 'oodInventorySha256': sha256_file(RUN_DIR / 'ood_inventory.json'),
    'photoReviewSha256': sha256_file(RUN_DIR / 'ood_review.csv'),
    'similarityReviewSha256': sha256_file(RUN_DIR / 'similarity_review.csv'),
    'oodImages': assigned_records, 'counts': dict(assignment_counts),
    'mangoInputManifestSha256': complete['inputManifestSha256'],
    'reuseStatus': 'Reused study photos. Report-only assignment in this run; not pristine external evidence.',
    'frozenAt': utc_now(),
}
write_json_once(RUN_DIR / 'evaluation_inputs.json', evaluation_inputs)
INPUT_HASH = sha256_file(RUN_DIR / 'evaluation_inputs.json')
write_csv_once(RUN_DIR / 'ood_assignments.csv', [
    {'id': record['id'], 'relativePath': record['relativePath'], 'subset': record['subset'],
     'groupId': record['groupId'], 'sha256': record['sha256']}
    for record in assigned_records
])
write_json_once(RUN_DIR / 'input_lock.json', {'evaluationInputsSha256': INPUT_HASH, 'createdAt': utc_now()})
tuning_records = [record for record in assigned_records if record['subset'] == 'ood_tune']
reporting_records = [record for record in assigned_records if record['subset'] == 'ood_report']
print('Frozen OOD group counts:', dict(assignment_counts))
print('No tuning or reporting photo belongs to both groups.')


## Step 7: Define consistent prediction and rejection calculations

**What this does:** Loads images exactly as the fresh model expects and defines confidence, margin, and entropy.

**Why this is needed:** Different preprocessing or applying temperature twice would make fitted thresholds describe the wrong model.

**How this works:** Confidence is the highest probability. Margin is the gap between the highest two. Normalized entropy measures how spread out all seven probabilities are, on a zero-to-one scale.

**Expected result:** The helpers are ready. Predictions require seven finite probabilities summing to one. No training or reporting predictions run in this step.


In [ ]:
%%mangoleaf
def verify_frozen_inputs():
    verify_artifacts()
    if sha256_file(RUN_DIR / 'evaluation_inputs.json') != INPUT_HASH:
        raise RuntimeError('Frozen evaluation inputs changed.')
    for filename, field in [
        ('ood_inventory.json', 'oodInventorySha256'),
        ('ood_review.csv', 'photoReviewSha256'),
        ('similarity_review.csv', 'similarityReviewSha256'),
    ]:
        if sha256_file(RUN_DIR / filename) != evaluation_inputs[field]:
            raise RuntimeError('A review or inventory changed after the split was frozen.')
    for record in ood_records:
        verify_photo(record, OOD_ROOT)


def has_webp_content(path):
    '''Read the file signature because a WebP photo may have a .jpg name.'''
    with Path(path).open('rb') as source:
        header = source.read(12)
    return header[:4] == b'RIFF' and header[8:12] == b'WEBP'


def load_pixels(path):
    # Match the training loader. MobileNet rescales internally; keep 0–255 input.
    if has_webp_content(path):
        with Image.open(path) as picture:
            pixels = np.asarray(ImageOps.exif_transpose(picture).convert('RGB'), dtype=np.float32)
    else:
        pixels = tf.io.decode_image(tf.io.read_file(str(path)), channels=3, expand_animations=False)
        pixels.set_shape((None, None, 3))
    return tf.image.resize(pixels, IMAGE_SIZE, method='bilinear', antialias=False)


def probability_stats(probabilities):
    result = []
    for row in probabilities:
        values = [float(value) for value in row]
        if len(values) != len(CLASS_NAMES) or any(not math.isfinite(value) or value < 0 or value > 1 for value in values):
            raise ValueError('Expected seven finite softmax probabilities in [0, 1].')
        if not math.isclose(sum(values), 1.0, abs_tol=1e-4):
            raise ValueError('Probabilities must sum to one. Do not pass logits.')
        ordered = sorted(values, reverse=True)
        result.append({
            'confidence': ordered[0], 'margin': ordered[0] - ordered[1],
            'entropy': -sum(value * math.log(value) for value in values if value > 0) / math.log(len(values)),
            'predictedIndex': values.index(ordered[0]),
            **{'probability_' + label: value for label, value in zip(CLASS_NAMES, values)},
        })
    return result


def weight_digest(model):
    digest = hashlib.sha256()
    for variable in model.weights:
        values = variable.numpy()
        digest.update(str(values.shape).encode())
        digest.update(str(values.dtype).encode())
        digest.update(values.tobytes())
    return digest.hexdigest()


def predict_records(model, records, root):
    predictions = []
    for start in range(0, len(records), BATCH_SIZE):
        batch_records = records[start:start + BATCH_SIZE]
        pixels = tf.stack([load_pixels(verify_photo(record, root)) for record in batch_records])
        values = model(pixels, training=False).numpy()
        if values.shape != (len(batch_records), len(CLASS_NAMES)):
            raise ValueError('The model did not return one seven-class prediction per image.')
        probability_stats(values)
        predictions.extend(values.tolist())
    if not predictions:
        raise ValueError('Cannot predict an empty image group.')
    return np.asarray(predictions, dtype=np.float32)


def accepted(stat, thresholds):
    return (
        stat['confidence'] >= thresholds['confidenceThreshold']
        and stat['margin'] >= thresholds['marginThreshold']
        and stat['entropy'] <= thresholds['entropyThreshold']
    )


def select_thresholds(validation_stats, tuning_stats, grid, confidence, minimum_retention):
    if not validation_stats or not tuning_stats or not 0 < minimum_retention <= 1:
        raise ValueError('Nonempty tuning groups and a valid retention target are required.')
    if not grid or not 0 <= confidence <= 1 or any(not math.isfinite(value) or not 0 <= value <= 1 for value in grid):
        raise ValueError('Threshold candidates must be finite numbers from zero to one.')
    required = math.ceil(minimum_retention * len(validation_stats) - 1e-9)
    best = None
    best_key = None
    search = []
    for margin in grid:
        validation_eligible = [row for row in validation_stats if row['confidence'] >= confidence and row['margin'] >= margin]
        tuning_eligible = [row for row in tuning_stats if row['confidence'] >= confidence and row['margin'] >= margin]
        for entropy in grid:
            retained = sum(row['entropy'] <= entropy for row in validation_eligible)
            rejected = len(tuning_stats) - sum(row['entropy'] <= entropy for row in tuning_eligible)
            feasible = retained >= required
            search.append({
                'margin': margin, 'entropy': entropy,
                'validationAccepted': retained, 'validationTotal': len(validation_stats),
                'validationRetention': retained / len(validation_stats),
                'oodRejected': rejected, 'oodTotal': len(tuning_stats),
                'oodRejectionRate': rejected / len(tuning_stats), 'meetsRetentionTarget': feasible,
            })
            # Exact counts decide ties. First reject more OOD; then retain more mango.
            # Further ties prefer a lower margin, then a higher maximum entropy.
            key = (rejected, retained, -margin, entropy)
            if feasible and (best_key is None or key > best_key):
                best_key = key
                best = {'confidenceThreshold': confidence, 'marginThreshold': margin, 'entropyThreshold': entropy}
    return best, search


def scored_records(records, statistics, thresholds):
    if len(records) != len(statistics):
        raise ValueError('Every photo must have exactly one prediction.')
    output = []
    for record, stat in zip(records, statistics):
        kept = accepted(stat, thresholds)
        predicted = CLASS_NAMES[stat['predictedIndex']]
        reasons = []
        if stat['confidence'] < thresholds['confidenceThreshold']:
            reasons.append('confidence')
        if stat['margin'] < thresholds['marginThreshold']:
            reasons.append('margin')
        if stat['entropy'] > thresholds['entropyThreshold']:
            reasons.append('entropy')
        output.append({
            'id': record['id'], 'relativePath': record['relativePath'], 'sha256': record['sha256'],
            'subset': record['subset'], 'actualClass': record.get('label', 'unsupported'),
            'rawPredictedClass': predicted, 'accepted': kept,
            'displayedClass': predicted if kept else 'rejected',
            'rejectionReasons': ';'.join(reasons), **stat,
        })
    return output


## Step 8: Fit settings using validation and OOD tuning photos only

**What this does:** Measures candidate rejection settings on the two groups reserved for selecting them.

**Why this is needed:** Reporting and mango test photos must not influence the choice.

**How this works:** Keeps confidence at 0.70; searches margin and maximum entropy from 0.00 through 1.00 in steps of 0.01. Requires at least 95% of mango validation predictions to remain accepted, including both correct and wrong predictions.

**Expected result:** All 10,201 search rows, tuning predictions, and the chosen settings are saved and locked before any reporting predictions.


**Selection order:** maximize the number of rejected OOD tuning photos while meeting
the validation retention requirement. If tied, accept more mango validation photos;
if still tied, choose the lower margin; if still tied, choose the higher maximum entropy.
These are declared choices, not proof that this recipe is optimal.

If no candidate retains enough mango validation photos, the notebook records an
**infeasible** selection and stops. Do not relax requirements after looking at reporting
results. A changed recipe needs a separately recorded experiment.


In [ ]:
%%mangoleaf
SELECTION_PATH = RUN_DIR / 'selected_thresholds.json'
if SELECTION_PATH.exists() or (RUN_DIR / 'evaluation_started.json').exists():
    raise RuntimeError('Selection already ran. Preserve this folder and start a new run to change the recipe.')
verify_frozen_inputs()

model = tf.keras.models.load_model(TRAINING_RUN / 'model_calibrated.keras', compile=False, safe_mode=True)
MODEL_WEIGHT_DIGEST = weight_digest(model)
validation_probabilities = predict_records(model, validation_records, SPLIT_DIR)
saved_validation = np.load(TRAINING_RUN / 'validation_probabilities.npy', allow_pickle=False)
if saved_validation.shape != validation_probabilities.shape:
    raise ValueError('Saved validation count/order differs from the fresh split.')
np.testing.assert_allclose(validation_probabilities, saved_validation, rtol=1e-4, atol=1e-4,
    err_msg='Validation parity failed. Check model identity and preprocessing before fitting.')
tuning_probabilities = predict_records(model, tuning_records, OOD_ROOT)
if weight_digest(model) != MODEL_WEIGHT_DIGEST:
    raise RuntimeError('Inference unexpectedly changed the model weights.')

validation_stats = probability_stats(validation_probabilities)
tuning_stats = probability_stats(tuning_probabilities)
chosen, search_rows = select_thresholds(
    validation_stats, tuning_stats, GRID,
    CONFIDENCE_THRESHOLD, MINIMUM_VALIDATION_RETENTION,
)
write_csv_once(RUN_DIR / 'threshold_search.csv', search_rows)
selection = {
    'experimentVersion': EXPERIMENT_VERSION, 'status': 'feasible' if chosen else 'infeasible',
    'classNames': CLASS_NAMES, 'thresholds': chosen,
    'trainingCompleteSha256': COMPLETE_HASH, 'evaluationInputsSha256': INPUT_HASH,
    'modelSha256': sha256_file(TRAINING_RUN / 'model_calibrated.keras'),
    'modelWeightDigest': MODEL_WEIGHT_DIGEST,
    'temperature': complete['calibratedTemperature'], 'temperatureAlreadyFolded': True,
    'confidenceFixed': CONFIDENCE_THRESHOLD, 'minimumValidationRetention': MINIMUM_VALIDATION_RETENTION,
    'gridStep': 0.01, 'searchRows': len(search_rows),
    'tieBreakOrder': ['more OOD rejected', 'more mango validation accepted', 'lower margin', 'higher maximum entropy'],
    'retentionIncludesIncorrectPredictions': True,
    'reportingUsedForFitting': False, 'testUsedForFitting': False,
    'reuseStatus': evaluation_inputs['reuseStatus'], 'frozenAt': utc_now(),
}
write_json_once(SELECTION_PATH, selection)
SELECTION_HASH = sha256_file(SELECTION_PATH)
write_json_once(RUN_DIR / 'selection_lock.json', {
    'selectionSha256': SELECTION_HASH, 'evaluationInputsSha256': INPUT_HASH,
    'trainingCompleteSha256': COMPLETE_HASH, 'frozenAt': utc_now(),
})
if chosen is None:
    raise RuntimeError('No feasible thresholds met the fixed validation retention target. Infeasibility was recorded; no reporting inference or candidate export will run.')

write_csv_once(RUN_DIR / 'validation_predictions.csv', scored_records(validation_records, validation_stats, chosen))
write_csv_once(RUN_DIR / 'ood_tuning_predictions.csv', scored_records(tuning_records, tuning_stats, chosen))
print('Frozen settings:', chosen)
print('All selection data is saved. Mango test and OOD reporting photos remain unused.')


## Step 9: Define reporting measures without hiding rejected mango photos

**What this does:** Defines raw classification scores and rejection-aware scores with clear denominators.

**Why this is needed:** Reporting accuracy only among accepted photos can hide how many supported photos were rejected.

**How this works:** Raw metrics use the highest-probability class before rejection. Rejection-aware recall and F1 retain every true mango photo in the denominator. Rejection is a separate output, never a healthy class.

**Expected result:** Metrics helpers are ready. Reporting will show total support, accepted counts, rejected counts, full-set correctness, and separately labelled accepted-only accuracy.


In [ ]:
%%mangoleaf
def classification_metrics(actual, predicted):
    metrics = []
    for label in CLASS_NAMES:
        true_positive = sum(truth == label and guess == label for truth, guess in zip(actual, predicted))
        false_positive = sum(truth != label and guess == label for truth, guess in zip(actual, predicted))
        false_negative = sum(truth == label and guess != label for truth, guess in zip(actual, predicted))
        support = sum(truth == label for truth in actual)
        precision = true_positive / (true_positive + false_positive) if true_positive + false_positive else 0.0
        recall = true_positive / (true_positive + false_negative) if true_positive + false_negative else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        metrics.append({
            'className': label, 'precision': precision, 'recall': recall, 'f1Score': f1,
            'support': support, 'truePositive': true_positive,
            'falsePositive': false_positive, 'falseNegative': false_negative,
        })
    return metrics


def summarize_mango(scored):
    if not scored:
        raise ValueError('A mango reporting subset cannot be empty.')
    actual = [row['actualClass'] for row in scored]
    raw_predicted = [row['rawPredictedClass'] for row in scored]
    selected_predicted = [row['displayedClass'] for row in scored]
    accepted_count = sum(row['accepted'] for row in scored)
    correct_raw = sum(truth == predicted for truth, predicted in zip(actual, raw_predicted))
    correct_accepted = sum(row['accepted'] and row['actualClass'] == row['rawPredictedClass'] for row in scored)
    raw_matrix = [[sum(truth == first and predicted == second for truth, predicted in zip(actual, raw_predicted)) for second in CLASS_NAMES] for first in CLASS_NAMES]
    rejection_matrix = [[sum(truth == first and predicted == second for truth, predicted in zip(actual, selected_predicted)) for second in CLASS_NAMES + ['rejected']] for first in CLASS_NAMES]
    return {
        'total': len(scored), 'accepted': accepted_count, 'rejected': len(scored) - accepted_count,
        'retention': accepted_count / len(scored), 'rawAccuracy': correct_raw / len(scored),
        'correctAcceptedFractionOfAllPhotos': correct_accepted / len(scored),
        'acceptedOnlyAccuracy': correct_accepted / accepted_count if accepted_count else None,
        'rawPerClassMetrics': classification_metrics(actual, raw_predicted),
        'rejectionAwarePerClassMetrics': classification_metrics(actual, selected_predicted),
        'rawConfusionMatrix': raw_matrix, 'rejectionAwareConfusionMatrix': rejection_matrix,
        'matrixRowLabels': CLASS_NAMES, 'rejectionMatrixColumnLabels': CLASS_NAMES + ['rejected'],
        'denominatorNote': 'All true mango photos remain in support. Rejected true mango photos count as missed class predictions.',
    }


def summarize_ood(scored):
    if not scored:
        raise ValueError('An OOD reporting subset cannot be empty.')
    false_accepts = sum(row['accepted'] for row in scored)
    return {
        'total': len(scored), 'falseAccepted': false_accepts, 'rejected': len(scored) - false_accepts,
        'falseAcceptRate': false_accepts / len(scored),
        'rejectionRate': (len(scored) - false_accepts) / len(scored),
    }


def read_selection_lock():
    frozen = read_json(SELECTION_PATH)
    lock = read_json(RUN_DIR / 'selection_lock.json')
    if sha256_file(SELECTION_PATH) != lock['selectionSha256'] or lock['selectionSha256'] != SELECTION_HASH:
        raise RuntimeError('The frozen rejection selection changed.')
    if frozen['status'] != 'feasible' or frozen['evaluationInputsSha256'] != INPUT_HASH:
        raise RuntimeError('No valid frozen gate belongs to these audited inputs.')
    if frozen['trainingCompleteSha256'] != COMPLETE_HASH or frozen['modelSha256'] != sha256_file(TRAINING_RUN / 'model_calibrated.keras'):
        raise RuntimeError('The frozen gate belongs to another model.')
    return frozen


## Step 10: Evaluate the frozen settings once

**What this does:** Runs the mango test and separate OOD reporting photos after the selection lock exists.

**Why this is needed:** These groups describe how frozen settings behave; they must not select the settings.

**How this works:** Creates an evaluation-started marker before prediction. Computes both confidence-only reference results and selected-gate results. The saved model fingerprint and weights are checked again.

**Expected result:** Prediction CSV files, raw and rejection-aware class metrics, evaluation.json, and an evaluation lock are saved. A repeated evaluation in this run is blocked.


In [ ]:
%%mangoleaf
if (RUN_DIR / 'evaluation_started.json').exists():
    raise RuntimeError('Evaluation already started in this folder. Preserve its evidence; do not retune or rerun it.')
frozen = read_selection_lock()
verify_frozen_inputs()
if weight_digest(model) != frozen['modelWeightDigest']:
    raise RuntimeError('In-memory model weights differ from the selected model.')
write_json_once(RUN_DIR / 'evaluation_started.json', {
    'selectionSha256': SELECTION_HASH, 'evaluationInputsSha256': INPUT_HASH,
    'trainingCompleteSha256': COMPLETE_HASH, 'startedAt': utc_now(),
})

test_probabilities = predict_records(model, test_records, SPLIT_DIR)
report_probabilities = predict_records(model, reporting_records, OOD_ROOT)
if weight_digest(model) != frozen['modelWeightDigest']:
    raise RuntimeError('Reporting inference changed the model weights.')

test_stats = probability_stats(test_probabilities)
report_stats = probability_stats(report_probabilities)
selected_test_rows = scored_records(test_records, test_stats, frozen['thresholds'])
selected_report_rows = scored_records(reporting_records, report_stats, frozen['thresholds'])
confidence_only = {'confidenceThreshold': frozen['confidenceFixed'], 'marginThreshold': 0.0, 'entropyThreshold': 1.0}
reference_test_rows = scored_records(test_records, test_stats, confidence_only)
reference_report_rows = scored_records(reporting_records, report_stats, confidence_only)

write_csv_once(RUN_DIR / 'mango_test_predictions.csv', selected_test_rows)
write_csv_once(RUN_DIR / 'ood_report_predictions.csv', selected_report_rows)
write_csv_once(RUN_DIR / 'confidence_only_mango_test_predictions.csv', reference_test_rows)
write_csv_once(RUN_DIR / 'confidence_only_ood_report_predictions.csv', reference_report_rows)

evaluation = {
    'experimentVersion': EXPERIMENT_VERSION, 'status': 'experimental; not adopted or deployed',
    'classNames': CLASS_NAMES, 'trainingRun': str(TRAINING_RUN),
    'trainingCompleteSha256': COMPLETE_HASH, 'selectionSha256': SELECTION_HASH,
    'evaluationInputsSha256': INPUT_HASH, 'thresholds': frozen['thresholds'],
    'mangoValidation': summarize_mango(scored_records(validation_records, validation_stats, frozen['thresholds'])),
    'oodTuning': summarize_ood(scored_records(tuning_records, tuning_stats, frozen['thresholds'])),
    'mangoTest': summarize_mango(selected_test_rows),
    'oodReport': summarize_ood(selected_report_rows),
    'confidenceOnlyReference': {
        'mangoTest': summarize_mango(reference_test_rows), 'oodReport': summarize_ood(reference_report_rows),
    },
    'reuseStatus': evaluation_inputs['reuseStatus'],
    'supportedDatasetStatus': 'Fresh grouped split of existing study mango photographs; not independently collected field accuracy.',
    'browserPredictionParityMeasured': False,
    'finishedAt': utc_now(),
}
write_csv_once(RUN_DIR / 'raw_per_class_metrics.csv', evaluation['mangoTest']['rawPerClassMetrics'])
write_csv_once(RUN_DIR / 'rejection_aware_per_class_metrics.csv', evaluation['mangoTest']['rejectionAwarePerClassMetrics'])
write_json_once(RUN_DIR / 'evaluation.json', evaluation)
EVALUATION_HASH = sha256_file(RUN_DIR / 'evaluation.json')
write_json_once(RUN_DIR / 'evaluation_lock.json', {
    'evaluationSha256': EVALUATION_HASH, 'selectionSha256': SELECTION_HASH,
    'evaluationInputsSha256': INPUT_HASH, 'finishedAt': utc_now(),
})

print('Raw mango test accuracy:', evaluation['mangoTest']['rawAccuracy'])
print('Mango acceptance fraction:', evaluation['mangoTest']['retention'])
print('Correct accepted / ALL mango photos:', evaluation['mangoTest']['correctAcceptedFractionOfAllPhotos'])
print('Accuracy among accepted mango photos ONLY:', evaluation['mangoTest']['acceptedOnlyAccuracy'])
print('Reused OOD reporting results:', evaluation['oodReport'])


## Step 11: Create clearly labelled figures from the new measurements

**What this does:** Creates new raw and rejection-aware matrices, class metric charts, and a threshold-selection chart.

**Why this is needed:** Figures should make mistakes, support counts, and the setting-selection tradeoff easy to inspect.

**How this works:** Uses only this run's saved predictions and search rows. Per-class bars show percentages and sample counts. The threshold chart shows an entropy slice at the selected margin, not the entire two-dimensional search.

**Expected result:** PNG figures appear under this run's figures folder. Titles begin with V4.4.0 Seven Class.


In [ ]:
%%mangoleaf
read_selection_lock()
if sha256_file(RUN_DIR / 'evaluation.json') != EVALUATION_HASH:
    raise RuntimeError('Reporting measurements changed before plotting.')

labels = [label.replace('_', ' ').title() for label in CLASS_NAMES]


def plot_matrix(matrix, column_labels, title, filename):
    values = np.asarray(matrix, dtype=int)
    figure, axis = plt.subplots(figsize=(11, 8))
    axis.imshow(values, cmap='Blues')
    for row in range(values.shape[0]):
        for column in range(values.shape[1]):
            color = 'white' if values[row, column] > values.max() / 2 else 'black'
            axis.text(column, row, str(values[row, column]), ha='center', va='center', color=color)
    axis.set_xticks(range(len(column_labels)), column_labels, rotation=35, ha='right')
    axis.set_yticks(range(len(labels)), labels)
    axis.set_xlabel('Predicted result')
    axis.set_ylabel('Actual class')
    axis.set_title(title)
    figure.tight_layout()
    figure.savefig(FIGURE_DIR / filename, dpi=180, bbox_inches='tight')
    plt.show()
    plt.close(figure)


plot_matrix(evaluation['mangoTest']['rawConfusionMatrix'], labels,
    'V4.4.0 Seven Class: Test Confusion Matrix (Before Rejection)', 'confusion_matrix.png')
plot_matrix(evaluation['mangoTest']['rejectionAwareConfusionMatrix'], labels + ['Rejected'],
    'V4.4.0 Seven Class: Test Confusion Matrix (With Rejection)', 'confusion_matrix_with_rejection.png')


def plot_class_metrics(rows, title, filename):
    figure, axis = plt.subplots(figsize=(12, 8))
    locations = np.arange(len(rows))
    for offset, field, color, label in [
        (-0.25, 'precision', '#2878b5', 'Precision'),
        (0.0, 'recall', '#f28e2b', 'Recall'),
        (0.25, 'f1Score', '#399c44', 'F1-score'),
    ]:
        percentages = [100 * row[field] for row in rows]
        bars = axis.barh(locations + offset, percentages, height=0.24, color=color, label=label)
        axis.bar_label(bars, labels=[f'{value:.1f}%' for value in percentages], padding=3, fontsize=9)
    axis.set_yticks(locations, [label + ' (n=' + str(row['support']) + ')' for label, row in zip(labels, rows)])
    axis.invert_yaxis()
    axis.set_xlim(0, 112)
    axis.set_xlabel('Score (%)')
    axis.set_title(title)
    axis.legend(loc='lower right')
    axis.grid(axis='x', alpha=0.2)
    figure.tight_layout()
    figure.savefig(FIGURE_DIR / filename, dpi=180, bbox_inches='tight')
    plt.show()
    plt.close(figure)


plot_class_metrics(evaluation['mangoTest']['rawPerClassMetrics'],
    'V4.4.0 Seven Class: Precision, Recall, and F1-Score (Before Rejection)', 'per_class_metrics.png')
plot_class_metrics(evaluation['mangoTest']['rejectionAwarePerClassMetrics'],
    'V4.4.0 Seven Class: Precision, Recall, and F1-Score (All Mango Photos, With Rejection)',
    'per_class_metrics_with_rejection.png')

chosen_margin = frozen['thresholds']['marginThreshold']
slice_rows = [row for row in search_rows if row['margin'] == chosen_margin]
figure, axis = plt.subplots(figsize=(11, 6))
axis.plot([row['entropy'] for row in slice_rows], [100 * row['validationRetention'] for row in slice_rows], label='Mango validation accepted')
axis.plot([row['entropy'] for row in slice_rows], [100 * row['oodRejectionRate'] for row in slice_rows], label='OOD tuning rejected')
axis.axhline(100 * frozen['minimumValidationRetention'], color='gray', linestyle='--', label=f"{100 * frozen['minimumValidationRetention']:.0f}% mango retention target")
axis.axvline(frozen['thresholds']['entropyThreshold'], color='crimson', linestyle=':', label='Selected maximum entropy')
axis.set_title('V4.4.0 Seven Class: Threshold Selection (Validation and Reused OOD Tuning)')
axis.set_xlabel('Maximum normalized entropy at margin = ' + f'{chosen_margin:.2f}' + ' and confidence = ' + f"{frozen['confidenceFixed']:.2f}")
axis.set_ylabel('Photos (%)')
axis.set_ylim(0, 103)
axis.legend()
axis.grid(alpha=0.2)
figure.tight_layout()
figure.savefig(FIGURE_DIR / 'threshold_search.png', dpi=180, bbox_inches='tight')
plt.show()
plt.close(figure)


## Step 12: Package the experimental browser model with its frozen settings

**What this does:** Copies the fresh TensorFlow.js export and adds the selected rejection settings to its configuration.

**Why this is needed:** The settings must travel with exactly the weights used for the measurements, without overwriting the application's adopted model.

**How this works:** Checks all browser shard paths and fingerprints, copies unchanged model bytes, and updates only the candidate configuration metadata. No additional calibration is performed.

**Expected result:** candidate_model contains a complete experimental export. Nothing is copied into the application's public/model folder.


In [ ]:
%%mangoleaf
read_selection_lock()
verify_frozen_inputs()
evaluation_lock = read_json(RUN_DIR / 'evaluation_lock.json')
if sha256_file(RUN_DIR / 'evaluation.json') != evaluation_lock['evaluationSha256']:
    raise RuntimeError('Evaluation records changed before packaging.')
if evaluation_lock['selectionSha256'] != SELECTION_HASH or evaluation_lock['evaluationInputsSha256'] != INPUT_HASH:
    raise RuntimeError('Evaluation does not belong to the frozen gate and inputs.')

source_export = TRAINING_RUN / 'tfjs_model'
browser_json = read_json(source_export / 'model.json')
required_browser_files = {'model.json', 'model_config.json'}
for group in browser_json['weightsManifest']:
    for relative in group['paths']:
        inside(source_export, relative)
        required_browser_files.add(relative)
if len(required_browser_files) < 3:
    raise ValueError('The browser export has no weight shards.')

candidate_root = RUN_DIR / 'candidate_model'
candidate_root.mkdir(exist_ok=False)
for relative in sorted(required_browser_files):
    expected_key = 'tfjs_model/' + relative
    if expected_key not in complete['artifacts']:
        raise ValueError('Browser file is absent from the completed training manifest: ' + relative)
    source = inside(source_export, relative)
    target = inside(candidate_root, relative)
    target.parent.mkdir(parents=True, exist_ok=True)
    if relative != 'model_config.json':
        shutil.copyfile(source, target)
        if sha256_file(target) != complete['artifacts'][expected_key]['sha256']:
            raise RuntimeError('A copied browser model file differs from the fresh training export.')

candidate_config = read_json(source_export / 'model_config.json')
if candidate_config.get('classes', candidate_config.get('classNames')) != CLASS_NAMES:
    raise ValueError('Browser configuration class order differs from the calibrated model.')
if candidate_config.get('temperatureFoldedIntoWeights') is not True:
    raise ValueError('Browser export must contain the already calibrated model.')
if not math.isclose(float(candidate_config['temperature']), float(complete['calibratedTemperature']), rel_tol=1e-12):
    raise ValueError('Browser export temperature metadata differs from the completed run.')
candidate_config.update({
    'modelVersion': 'model-v4.4.0-candidate', 'experimentVersion': EXPERIMENT_VERSION,
    'thresholds': frozen['thresholds'], 'status': 'experimental candidate; not adopted or deployed',
    'selectionSha256': SELECTION_HASH, 'evaluationSha256': EVALUATION_HASH,
    'trainingCompleteSha256': COMPLETE_HASH, 'browserPredictionParityMeasured': False,
})
write_json_once(candidate_root / 'model_config.json', candidate_config)
candidate_hashes = {
    path.relative_to(candidate_root).as_posix(): {'sha256': sha256_file(path), 'bytes': path.stat().st_size}
    for path in sorted(candidate_root.rglob('*')) if path.is_file()
}
write_json_once(RUN_DIR / 'candidate_hashes.json', candidate_hashes)

with zipfile.ZipFile(RUN_DIR / 'candidate_model.zip', 'x', compression=zipfile.ZIP_DEFLATED) as archive:
    for relative in sorted(candidate_hashes):
        archive.write(inside(candidate_root, relative), arcname=relative)
print('Experimental browser export:', candidate_root)
print('Application model adoption requires a separate decision and applicable evaluation.')


## Step 13: Save the final evidence manifest and read the result limits

**What this does:** Writes a completion record tied to every file this threshold run produced.

**Why this is needed:** A reviewer needs to distinguish executed measurements, experimental exports, and evidence still outside this notebook's scope.

**How this works:** Rechecks frozen inputs and output locks, fingerprints the artifacts, and records the reporting scope in plain English.

**Expected result:** threshold_fitting_complete.json exists. Keep this run folder and the executed notebook together as experiment evidence.


In [ ]:
%%mangoleaf
read_selection_lock()
verify_frozen_inputs()
if sha256_file(RUN_DIR / 'evaluation.json') != EVALUATION_HASH:
    raise RuntimeError('Final reporting evidence changed.')
artifacts = {
    path.relative_to(RUN_DIR).as_posix(): {'sha256': sha256_file(path), 'bytes': path.stat().st_size}
    for path in sorted(RUN_DIR.rglob('*')) if path.is_file()
}
write_json_once(RUN_DIR / 'threshold_fitting_complete.json', {
    'schemaVersion': 1, 'experimentVersion': EXPERIMENT_VERSION,
    'status': 'completed threshold experiment; not adopted or deployed',
    'trainingRun': str(TRAINING_RUN), 'trainingCompleteSha256': COMPLETE_HASH,
    'selectionSha256': SELECTION_HASH, 'evaluationSha256': EVALUATION_HASH,
    'evaluationInputsSha256': INPUT_HASH, 'classNames': CLASS_NAMES,
    'thresholds': frozen['thresholds'], 'oodCounts': dict(assignment_counts),
    'modelWeightsChangedByThisNotebook': False,
    'reuseStatus': evaluation_inputs['reuseStatus'],
    'browserPredictionParityMeasured': False,
    'artifacts': artifacts, 'finishedAt': utc_now(),
})
print('Completed experiment:', RUN_DIR)
print('Save the executed notebook with this run. Its original template contains no claimed results.')


## Reading the results honestly

- **Raw mango accuracy** measures the highest-probability class on this saved mango test split.
- **Retention** is the fraction of true mango photos accepted, whether correctly or incorrectly.
- **Correct accepted / all photos** uses the full mango test count as its denominator.
- **Accepted-only accuracy** describes a smaller accepted subset and must be labelled that way.
- **OOD rejection rate** describes rejection on the reused unsupported reporting group. It is
  not an estimate of all unrelated photographs a farmer might encounter.
- No class has a general guarantee because this split reports a high or even 100% score.
  Read its support count and mistakes along with precision, recall, and F1-score.
- TensorFlow.js model bytes are preserved from the fresh training export. This notebook does
  not measure browser prediction parity. Converter success alone does not establish parity.
- The separate OOD groups protect this run's selection process, but earlier study use of
  the same photos remains a limitation. Keep that history visible in the experiment report.

## Primary documentation

- [TensorFlow image decoding](https://www.tensorflow.org/api_docs/python/tf/io/decode_image)
- [Keras model loading](https://keras.io/api/models/model_saving_apis/model_saving_and_loading/)
- [Keeping related samples together](https://scikit-learn.org/stable/modules/cross_validation.html#cross-validation-iterators-for-grouped-data)
- [Separating threshold tuning from evaluation](https://scikit-learn.org/stable/modules/classification_threshold.html)
- [Definitions of precision, recall, and F1](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_recall_fscore_support.html)

The threshold-tuning reference explains the separation principle. Its binary classifier
utility is not used as a replacement for this seven-class rejection search.
